# Statistics for RL — Drills Notebook

This notebook accompanies the [Statistics for RL](https://codefrydev.in/Reinforcement/math-for-rl/statistics/) page.
Practice computing mean, variance, standard error, histograms, and correlation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Mean and Standard Deviation

In [ ]:
# Episode returns from 20 evaluation runs
np.random.seed(42)
episode_returns = np.random.normal(loc=250, scale=40, size=20)

mean = np.mean(episode_returns)
std = np.std(episode_returns, ddof=1)  # unbiased (n-1)
n = len(episode_returns)
se = std / np.sqrt(n)

print(f'n = {n}')
print(f'Mean:   {mean:.2f}')
print(f'Std:    {std:.2f}')
print(f'SE:     {se:.2f}')
print(f'Report: {mean:.1f} ± {se:.2f} (SE)')


## 2. Histogram of episode returns

In [ ]:
np.random.seed(42)
n_episodes = 200
returns_normal = np.random.normal(250, 40, n_episodes)

# Bimodal distribution (unstable policy)
returns_bimodal = np.concatenate([
    np.random.normal(50, 10, n_episodes // 2),
    np.random.normal(450, 10, n_episodes // 2)
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(returns_normal, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(returns_normal.mean(), color='red', linestyle='--', label=f'mean={returns_normal.mean():.0f}')
axes[0].set_title('Stable policy (unimodal)'); axes[0].legend()

axes[1].hist(returns_bimodal, bins=20, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(returns_bimodal.mean(), color='red', linestyle='--', label=f'mean={returns_bimodal.mean():.0f}')
axes[1].set_title('Unstable policy (bimodal — mean is misleading)'); axes[1].legend()

for ax in axes:
    ax.set_xlabel('Episode return'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()


## 3. Standard Error vs Number of Evaluation Episodes

In [ ]:
true_mean = 250
true_std = 40
np.random.seed(42)

n_values = [3, 5, 10, 20, 50, 100]
se_values = [true_std / np.sqrt(n) for n in n_values]

plt.figure(figsize=(8, 4))
plt.plot(n_values, se_values, 'o-', color='steelblue')
plt.xlabel('Number of evaluation episodes (n)')
plt.ylabel('Standard Error of Mean')
plt.title('SE decreases as 1/√n — use more evaluation episodes')
plt.grid(alpha=0.3); plt.show()

for n, se in zip(n_values, se_values):
    print(f'n={n:3d}: SE = {se:.2f}')


## 4. Correlation: reward vs episode length

In [ ]:
np.random.seed(42)
n = 50
lengths = np.random.randint(10, 100, n)
# Positive correlation: longer episodes → higher rewards
rewards = 3 * lengths + np.random.randn(n) * 20

r = np.corrcoef(lengths, rewards)[0, 1]

print(f'Pearson r = {r:.4f}')
plt.scatter(lengths, rewards, alpha=0.6, color='steelblue')
m, b = np.polyfit(lengths, rewards, 1)
x_line = np.linspace(lengths.min(), lengths.max(), 100)
plt.plot(x_line, m * x_line + b, 'r-', label=f'r = {r:.3f}')
plt.xlabel('Episode length'); plt.ylabel('Reward')
plt.title('Correlation between reward and episode length')
plt.legend(); plt.show()


## 5. Bootstrapped confidence interval

In [ ]:
np.random.seed(42)
samples = np.array([230, 245, 255, 240, 270, 235, 260, 250, 265, 248])
n_boot = 10000

boot_means = np.array([
    np.mean(np.random.choice(samples, size=len(samples), replace=True))
    for _ in range(n_boot)
])

ci_lower = np.percentile(boot_means, 2.5)
ci_upper = np.percentile(boot_means, 97.5)

print(f'Sample mean: {np.mean(samples):.1f}')
print(f'Bootstrap 95% CI: [{ci_lower:.1f}, {ci_upper:.1f}]')

plt.hist(boot_means, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(ci_lower, color='red', linestyle='--', label=f'2.5th={ci_lower:.1f}')
plt.axvline(ci_upper, color='red', linestyle='--', label=f'97.5th={ci_upper:.1f}')
plt.axvline(np.mean(samples), color='green', label=f'mean={np.mean(samples):.1f}')
plt.title('Bootstrap distribution of sample mean'); plt.legend(); plt.show()
